In [2]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from dataclasses import dataclass

torch.manual_seed(1024)

d:\anaconda3\envs\tf2\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
@dataclass  #有了 @dataclass 可以省略__init__, 简写class
class GPTConfig:
    block_size: int=512
    batch_size: int=16
    n_layer: int=6
    n_head: int=12
    n_embd: int=768
    # head_size: int=n_embd // n_head
    dropout: float=0.1
    vocab_size: int=50257

In [ ]:
class Config:
    def __init__(self, block_size: int=512,
    batch_size: int=16,
    n_layer: int=6,
    n_head: int=12,
    n_embd: int=768,
    dropout: float=0.1,
    vocab_size: int=50257):
        
        self.block_size = block_size
        self.batch_size = batch_size
        self.n_embd = n_embd
        self.head_size = n_embd // n_head

In [ ]:
class SingleHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.q_proj = nn.Linear(config.n_embd, config.head_size)
        self.k_proj = nn.Linear(config.n_embd, config.head_size)
        self.v_proj = nn.Linear(config.n_embd, config.head_size)
        self.register_buffer(
            'attention_mask',
            torch.tril(torch.ones(config.block_size, config.block_size))
        )
        self.head_dim = config.head_size
        self.dropout = nn.Dropout(config.dropout)

        def forward(self, x):
            batch_size, seq_len, dim = x.size()
            q = self.q_proj(x)
            k = self.k_proj(x)
            v = self.v_proj(x)
            weight = torch.matmul(q, k.transpose(-1, -2)) / math.sqrt(self.head_dim)
            weight = weight.masked_fill(
                self.attention_mask[:seq_len, :seq_len] == 0,
                float('-inf')
            ) 
            weight = F.softmax(weight, dim=-1)
            weight = self.dropout(weight)
            output = weight @ v
            return output

# class MultiHeadAttention(nn.Module):
#     def __init__(self, config):
#         super().__init__()
#         self.q_proj = nn.Linear(config.n_embd, config.n_embd)
#         self.k_proj = nn.Linear(config.n_embd, config.n_embd)
#         self.v_proj = nn.Linear(config.n_embd, config.n_embd)
#         self.register_buffer(
#             'attention_mask',
#             torch.tril(torch.ones(config.block_size, config.block_size))
#         )
#         self.head_dim = config.n_embd // config.n_head
#         self.n_head = config.n_head
#         self.dropout = nn.Dropout(config.dropout)
#         self.out_proj = nn.Linear(config.n_embd, config.n_embd)

#     def forward(self, x):
#         batch_size, seq_len, dim = x.shape
#         q = self.q_proj(x)
#         k = self.k_proj(x)
#         v = self.v_proj(x)

#         q = q.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
#         k = k.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
#         v = v.view(batch_size, seq_len, self.n_head, self.head_dim).transpose(1, 2)
#         weight = q @ k.transpose(-1, -2) / math.sqrt(self.head_dim)
#         weight = weight.masked_fill(
#             self.attention_mask[:seq_len, :seq_len] == 0,
#             float('-inf')
#         )
#         weight = self.dropout(F.softmax(weight, dim=-1))
#         output_mid = weight @ v
#         output_mid = output_mid.transpose(1, 2).contiguous().view(batch_size, seq_len, -1)
#         output = self.out_proj(output_mid)
#         return output


# config = GPTConfig(n_embd=128, n_head=8, block_size=256, dropout=0.1)
# model = MultiHeadAttention(config)

# # 输入: [batch=3, seq_len=10, n_embd=128]
# X = torch.rand(3, 10, 128)

# output = model(X)
# output

class MultiHeadAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                SingleHeadAttention(config)
                for _ in range(config.n_head)
            ]
        )
        self.proj = nn.Linear(config.n_embd, config.n_embd)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x)
        output = torch.cat(
            [h(x) for h in self.heads],
            dim=-1
        )
        output = self.proj(output)
        output = self.dropout(output)
        return output



class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(config.n_embd, 4 * config.n_embd),
            nn.GELU(),
            nn.Linear(4 * config.n_embd, config.n_embd),
            nn.Dropout(config.dropout)
        )
    
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.head_size = config.n_embd // config.n_head
        self.att = MultiHeadAttention(config)
        self.ffn = FeedForward(config)
        self.ln1 = nn.LayerNorm(config.n_embd)
        self.ln2 = nn.LayerNorm(config.n_embd)

    def forward(self, x):
        x = x + self.att(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x
    
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.token_embedding_table = nn.Embedding(config.vocab_size, config.n_embd)
        self.position_embedding_table = nn.Embedding(config.block_size, config.n_embd)
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config.n_layer)]
        )
        self.ln_final = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.token_embedding_table.weight = self.lm_head.weight
        self.block_size = config.block_size
        
    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
    
    def forward(self, idx, targets=None):
        batch, seq_len = idx.shape
        token_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(
            torch.arange(seq_len, device=idx.device)
        )
        x = token_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_final(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            batch, seq_len, vocab_size = logits.size()
            logits = logits.view(batch * seq_len, vocab_size)
            targets = targets.view(batch * seq_len)
            loss = F.cross_entropy(logits, targets)
        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.block_size else idx[:, -self.block_size]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
class Expert(nn.Module):
    def __init__(self, feature_in, feature_out):
        super().__init__()
        self.linear = nn.Linear(feature_in, feature_out)

    def forward(self, x):
        return self.linear(x)

class BasicMOE(nn.Module):
    def __init__(self, feature_in, feature_out, n_expert):
        super().__init__()
        self.gate = nn.Linear(feature_in, n_expert)
        self.experts = nn.ModuleList(
            [Expert(feature_in, feature_out) for _ in range(n_expert)]
        )

    def forward(self, x):  # x shape (batch, feature_in)
        expert_weight = self.gate(x)  # (batch, n_expert)
        expert_out = [expert(x).unsqueeze(1) for expert in self.experts]  #(batch, 1, f_out)
        expert_out = torch.cat(expert_out, dim=1) #(batch, n_expert, f_out)
        expert_weight = F.softmax(expert_weight.unsqueeze(1),
                                  dim=-1) #(batch, 1, n_expert)
        return expert_weight @ expert_out
        

In [38]:
x = torch.tensor([
    [1, 5, 3, 2],
    [4, 7, 1, 8],
    [3, 2, 9, 6]
])
print(f"张量:\n{x}")

# 沿着最后一维（每行）取最大的2个
values, indices = torch.topk(x, k=2, dim=-1)


mask = F.one_hot(
    indices,
    num_classes=4
)
mask


张量:
tensor([[1, 5, 3, 2],
        [4, 7, 1, 8],
        [3, 2, 9, 6]])


tensor([[[0, 1, 0, 0],
         [0, 0, 1, 0]],

        [[0, 0, 0, 1],
         [0, 1, 0, 0]],

        [[0, 0, 1, 0],
         [0, 0, 0, 1]]])

In [36]:
indices

tensor([[1, 2],
        [3, 1],
        [2, 3]])

In [39]:
x = torch.tensor([[1, 2], [3, 4]])
condition = x > 2

indices = torch.where(condition)

In [41]:
indices

(tensor([1, 1]), tensor([0, 1]))

In [42]:
x

tensor([[1, 2],
        [3, 4]])